In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage, ToolMessage
from typing import Annotated
import requests
import os

In [ ]:
# Create tools

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    \"\"\"Fetch the currency conversion factor between two currencies.\"\"\"

    url = f"https://v6.exchangerate-api.com/v6/{os.getenv('EXCHANGE_RATE_API_KEY')}/pair/{base_currency}/{target_currency}"
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    data = response.json()

    if data.get("result") != "success":
        raise ValueError(data.get("error-type", "Currency API request failed"))

    return data["conversion_rate"]


@tool
def convert(
    base_currency_value: float,
    conversion_rate: Annotated[float, InjectedToolArg]
) -> float:
    \"\"\"Convert a base-currency amount using the injected exchange rate.\"\"\"\"

    return base_currency_value * conversion_rate

### Why `InjectedToolArg` is used
`conversion_rate` is marked as `InjectedToolArg`, so it is hidden from the LLM's tool-call schema. It is intended to be supplied by the code that executes the tool, not by the LLM.

**Important:** `bind_tools()` only exposes tools to the model. It does not automatically execute tools or transfer the output of one tool into another. The reliable sequential example at the bottom fetches the rate first, then passes it into `convert`. 

In [ ]:
get_conversion_factor.args

In [ ]:
convert.args

In [ ]:
# Test the exchange-rate tool directly (optional)
get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})

In [ ]:
# Test the conversion tool directly (optional)
convert.invoke({"base_currency_value": 10, "conversion_rate": 92.753})

In [ ]:
# Initialize the LLM (set GROQ_API_KEY in your environment)
llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
# Start a conversation
messages = [
    HumanMessage(
        "Get the conversion rate from INR to USD, then use the convert tool "
        "to convert 10 INR to USD. Use both tools."
    )
]

In [ ]:
messages

In [ ]:
# Ask the model for tool-call requests
ai_message = llm_with_tools.invoke(messages)
ai_message

In [ ]:
# Add the AI response to the conversation
messages.append(ai_message)

In [ ]:
# Inspect the requested tool calls (the model may request one or both)
ai_message.tool_calls

In [ ]:
# Execute tool calls, and if the rate tool was requested, use its result to run convert.
# This notebook handles the sequential case as well as the case where both calls are requested.

tool_messages = []
conversion_rate = None
amount_to_convert = 10

for tool_call in ai_message.tool_calls:
    if tool_call["name"] == "get_conversion_factor":
        result = get_conversion_factor.invoke(tool_call["args"])
        conversion_rate = result
        tool_messages.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )

    elif tool_call["name"] == "convert":
        amount_to_convert = tool_call["args"].get("base_currency_value", amount_to_convert)
        # conversion_rate may not be available yet if the model requested both tools at once.
        if conversion_rate is None:
            raise ValueError(
                "The convert tool was requested before the conversion rate was retrieved. "
                "Run the exchange-rate tool first, then invoke convert with the rate injected."
            )
        result = convert.invoke({
            "base_currency_value": amount_to_convert,
            "conversion_rate": conversion_rate
        })
        tool_messages.append(
            ToolMessage(content=str(result), tool_call_id=tool_call["id"])
        )

messages.extend(tool_messages)

In [ ]:
messages

In [ ]:
# Ask the LLM to respond based on tool results
llm_with_tools.invoke(messages).content

In [ ]:
# Reliable sequential workflow: get the rate, then calculate with the second tool.
# This does not depend on the model requesting both tools in the same AIMessage.

rate = get_conversion_factor.invoke({"base_currency": "INR", "target_currency": "USD"})
final_conversion = convert.invoke({
    "base_currency_value": 10,
    "conversion_rate": rate
})

print(f"Conversion rate (INR -> USD): {rate}")
print(f"10 INR converted to USD: {final_conversion}")